# Operational Intelligence
### A local-first reporting pipeline for operational KPIs

**Reproducible portfolio demonstration** · Python · SQLite · CSV/Excel · executive analytics

This notebook accompanies the original desktop application's refactored source modules (`core.py`, `collector.py`, `desktop.py`). It demonstrates the database and analytical pipeline **without starting a GUI, browser, or external API**. The source files provided for this project are blank templates; all observations in the committed sample database are **synthetic**.

> Privacy boundary: do not commit live chat messages, staff identifiers, real attendance reports, or WhatsApp sessions. Run this notebook only with the included sample data.


## 1 — Reproducible environment

Install Python 3.10+ and the dependencies from the repository root: `python -m pip install -r requirements.txt`. For Jupyter, install `jupyterlab` or `ipykernel` as required. The core SQLite and CSV logic itself uses only the Python standard library.


In [ ]:
from pathlib import Path
import csv
import sqlite3
import sys

REPO = Path.cwd().resolve()
if not (REPO / 'core.py').exists():
    REPO = REPO.parent
assert (REPO / 'core.py').exists(), 'Open notebook from inside the downloaded repository.'
sys.path.insert(0, str(REPO))

import core

demo_db = REPO / 'examples' / 'demo_operations.db'
assert demo_db.exists(), 'The synthetic example DB must be present.'
print('Repository:', REPO.name)
print('Synthetic database:', demo_db.relative_to(REPO))
print('Live WhatsApp collection: OFF')


## 2 — Verify the two supplied source structures

The original Excel and CSV files both contain the 11 headings below. The stored CSV template has been trimmed of blank preformatted lines. The Excel template preserves the uploaded workbook structure. Source data dates are interpreted as **DD/MM/YYYY**, while incoming WhatsApp metadata is configured separately as **MM/DD/YYYY**.


In [ ]:
csv_template = REPO / 'templates' / 'WhatsAppData_staging.csv'
excel_template = REPO / 'templates' / 'WhatsAppData.xlsx'
with csv_template.open(newline='', encoding='utf-8-sig') as handle:
    csv_headers = next(csv.reader(handle))
assert csv_headers == core.FIELDS
print('Input columns:', ', '.join(core.FIELDS))
print('CSV template bytes:', csv_template.stat().st_size)
print('Excel template bytes:', excel_template.stat().st_size)
print('Note: both templates contain zero operational records.')


## 3 — SQLite schema and data integrity

The database schema is created by `core.init_db()`; `reports` stores canonical messages, `alerts` stores exceptions, `audit` stores lineage events, and `monitor_state` tracks collector status. Reports have a unique message fingerprint. This cell opens the committed **synthetic** DB in **read-only mode**.


In [ ]:
uri = f'file:{demo_db.resolve().as_posix()}?mode=ro'
with sqlite3.connect(uri, uri=True) as conn:
    conn.row_factory = sqlite3.Row
    tables = [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")]
    integrity = conn.execute('PRAGMA integrity_check').fetchone()[0]
    records = [dict(r) for r in conn.execute('SELECT * FROM reports ORDER BY event_at DESC')]
print('Tables:', ', '.join(tables))
print('SQLite integrity:', integrity)
print('Synthetic messages:', len(records))
assert integrity == 'ok' and len(records) == 168
assert {r['origin'] for r in records} == {'demo'}


## 4 — Executive KPI overview

The summary tracks volume, number of reporting teams, late counts, reported near-miss values and incomplete safety fields. Unknown values must **not** be interpreted as zeros. Values are simulated operational **reports**, not verified attendance or incident records.


In [ ]:
kpis = core.summary(records, expected_groups=['Gang 1', 'Team 5'])
for label, value in [
    ('Reports received', kpis['reports']),
    ('Reporting teams', kpis['teams']),
    ('Late values reported', kpis['late']),
    ('Near-miss values reported', kpis['near_miss']),
    ('Incomplete safety records', kpis['missing_safety']),
    ('Coverage of configured demo teams', f"{kpis['coverage']:.0f}%"),
]:
    print(f'{label:38} {value}')


## 5 — Reported near-miss trend and team coverage

The plot includes **only dates that actually contain reports**. Unreported days are not added or represented as zero-valued observations.


In [ ]:
import matplotlib.pyplot as plt

daily = core.safety_daily(records)
dates = list(daily)
values = [daily[day]['near_miss'] for day in dates]
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(dates, values, label='Reported near-miss values')
ax.set_title('Synthetic demonstration — daily reported near misses')
ax.set_xlabel('Report date')
ax.set_ylabel('Reported count')
ax.set_xticks(dates[::max(1, len(dates) // 9)])
ax.tick_params(axis='x', rotation=35)
ax.grid(axis='y', alpha=0.2)
fig.tight_layout()
plt.show()


## 6 — Validate round-trip import and deduplication, in a disposable database

This demonstration uses Python's temporary directory and never modifies the committed example database or any real local database. The synthetic CSV and workbook contain the same 168 messages and the exact 11 headings of the original templates.


In [ ]:
from tempfile import TemporaryDirectory

old_db = core.DB
try:
    with TemporaryDirectory() as temp:
        core.DB = Path(temp) / 'test.db'
        core.init_db()
        added_csv, duplicated_csv, rejected_csv = core.import_csv(REPO / 'examples' / 'demo_WhatsAppData.csv')
        added_xlsx, duplicated_xlsx, rejected_xlsx = core.import_excel(REPO / 'examples' / 'demo_WhatsAppData.xlsx')
        n = len(core.filtered())
        print('CSV first import: ', added_csv, 'inserted;', duplicated_csv, 'duplicates')
        print('XLSX second import:', added_xlsx, 'inserted;', duplicated_xlsx, 'duplicates')
        print('Consolidated record count:', n)
        assert (added_csv, added_xlsx, duplicated_xlsx, n) == (168, 0, 168, 168)
        assert not rejected_csv and not rejected_xlsx
finally:
    core.DB = old_db


## 7 — Design decisions and next steps

**Existing functions retained:** local SQLite and audit trail, CSV/Excel history, browser monitoring (opt-in), dashboard charting, alerts, time/group filters, light/dark theme, exports and database relocation. The application runs locally with `python app.py`; it should not be started inside this notebook because it starts a desktop GUI event loop.

**Engineering trade-offs:** WhatsApp Web scraping is UI-dependent, needs permission and can break when selectors change. Reporting metrics need domain definitions, missing-value handling, independently verified safety data and robust production data governance. There is no external API, cloud database or claim that message-reported figures are validated incidents.

**Further enhancements:** support the additional change-before-3pm / change-after-3pm, no-show and shortage workflows only after obtaining those actual schemas and business rules; define source precedence, effective dates, audit events and reconciliation ownership.

See `README.md`, `docs/data_dictionary.md`, `docs/architecture.md` and `SECURITY.md`. Do not publish employer IP or real operational data without authorization.
